In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from openai import OpenAI

client = OpenAI()

In [2]:
def ask_llm(messages, temperature=0.3, max_tokens=500):
    """메시지 리스트를 받아서 LLM의 답변 텍스트를 돌려준다."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

---

# 🎁 도전과제

이론 실습은 끝났습니다. 이제 배운 걸로 **실무에서 자주 만나는** 작업을 직접 만들어봅시다.

## 도전과제 1. 회의록 요약 — system 메시지 선택하기

긴 회의록을 받아서 3줄 요약을 돌려주는 비서를 만들어야 합니다.  
빈칸을 채우는 게 아니라 **3가지 system 메시지 후보 중 어느 게 가장 좋은가** 판단해보세요.

- 셋 다 돌려보고, 결과를 비교한 뒤 *근거를 들어* 한 개를 고릅니다.
- 이게 실무에서 프롬프트를 설계하는 진짜 방식이에요.

In [7]:
candidates = {
    "A": "회의록을 요약하세요.",
    "B": "당신은 회의록 요약 비서입니다. 3줄로 요약하세요.",
    "C": ("당신은 회의록 요약 비서입니다. 정확히 3줄로 요약하되, "
          "각 줄은 (결정사항) / (담당자) / (마감일) 형식을 따르세요. "
          "회의록에 없는 내용은 절대 추측하지 마세요."),
}

sample_notes = """
[2026-05-14 팀 미팅]
- 김PM: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안. 이유는 평가 데이터 부족.
- 이개발: 6월 15일이면 가능. 다만 RAG 평가셋 100건 추가 수집 필요.
- 박마케팅: 출시 연기 시 광고 일정도 조정해야 함. 다음 주 월요일까지 결정 필요.
"""

# 세 후보를 모두 실행해서 비교
for key, system_msg in candidates.items():
    result = ask_llm([
        {"role": "system", "content": system_msg},
        {"role": "user", "content": sample_notes},
    ], temperature=0)
    print(f"=== 후보 {key} ===")
    print(result)
    print()

=== 후보 A ===
**회의록 요약 (2026-05-14 팀 미팅)**

- **김PM**: 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기 제안, 이유는 평가 데이터 부족.
- **이개발**: 6월 15일 출시 가능하나, RAG 평가셋 100건 추가 수집 필요.
- **박마케팅**: 출시 연기 시 광고 일정 조정 필요, 다음 주 월요일까지 결정 필요.

=== 후보 B ===
김PM은 신규 챗봇 출시일을 6월 1일에서 6월 15일로 연기할 것을 제안했으며, 이개발은 추가 데이터 수집이 필요하다고 언급했다. 박마케팅은 출시 연기에 따라 광고 일정 조정이 필요하다고 강조하며, 다음 주 월요일까지 결정을 요구했다. 팀은 출시 일정과 관련된 사항을 조율하기로 했다.

=== 후보 C ===
(신규 챗봇 출시일을 6월 15일로 연기) / (김PM, 이개발, 박마케팅) / (다음 주 월요일)



1. 실무에서 쓴다면 후보 B가 가장 쓸 만하다.

2. 연기 출시일을 명시하고, 이후 연기 사유를 알려주며 연기에 따른 조정 사항을 고지하고 고지 사항에 대한 결정일을 명시하고 최종적으로 회의록의 주요 논점을 알려주기 때문이다.

## 도전과제 2. 고객 문의 자동 분류기

고객 문의 한 줄을 받아서 **`결제`, `배송`, `환불`, `기타`** 중 하나로 분류하세요.

이제 여러분은 *왜* `temperature=0`, `max_tokens=10`이 필요한지 압니다 (위의 🐛 버그 셀 기억나죠?).

**힌트**:
- `system` 메시지에 "다음 4개 중 하나로만 답하세요"라고 명시
- `temperature=0` (일관성 필수), `max_tokens=10` (단어 하나면 충분)

In [5]:
def classify_inquiry(text: str) -> str:
    """고객 문의를 4개 카테고리 중 하나로 분류"""
    # 👇 여기를 채워보세요
    messages = [
        {"role": "system", "content": "결제/배송/환불/기타 중 하나로만 대답해"},  # ← 빈칸
        {"role": "user", "content": text},
    ]
    return ask_llm(messages, temperature=0, max_tokens=10)


# 🎯 실행 *전* 예측: 아래 4개 문의가 어떻게 분류될지 머릿속으로 먼저 답을 적어보세요
test_cases = [
    "어제 주문했는데 카드 결제가 두 번 되었어요.",       # 예측: 결제
    "택배가 일주일째 안 와요.",                          # 예측: 배송
    "사이즈가 안 맞아서 반품하고 싶어요.",                # 예측: 환불
    "영업시간이 어떻게 되나요?",                         # 예측: 기타
]

# 이제 실행해서 예측과 비교
for text in test_cases:
    category = classify_inquiry(text)
    print(f"[{category:>4}] {text}")

[  결제] 어제 주문했는데 카드 결제가 두 번 되었어요.
[  배송] 택배가 일주일째 안 와요.
[  환불] 사이즈가 안 맞아서 반품하고 싶어요.
[  기타] 영업시간이 어떻게 되나요?
